In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve

import tensorflow as tf
import tf_keras
from tf_keras import layers, Model, regularizers
from tf_keras.callbacks import EarlyStopping

from Functions.distance import euclidean_distance, contrastive_loss

## Costanti

In [ ]:
N_VAL_SUBJECTS = 4     # Fig. 9 del paper: 116 train / 4 val / 1 test
AUG_FRACTION = 0.2
CONTRASTIVE_MARGIN = 1.0

N_DEBUG_FOLDS = 16     # per il debug rapido; None per saltarlo
DEBUG_EPOCHS = 40      # alzato da 20: la contrastive loss ha bisogno di piu' tempo
DEBUG_STEPS_PER_EPOCH = 80
DEBUG_PATIENCE = 8

FULL_EPOCHS = 50
FULL_STEPS_PER_EPOCH = 100
FULL_PATIENCE = 6

## Caricamento dati precalcolati

In [ ]:
def load_precomputed(power_tensor_path="power_tensor.npy",
                      labels_path="labels.npy",
                      subject_ids_path="subject_ids.npy",
                      brain_maps_path="brain_maps.npy"):
    power_tensor = np.load(power_tensor_path)
    labels = np.load(labels_path)
    subject_ids = np.load(subject_ids_path, allow_pickle=True)
    brain_maps = np.load(brain_maps_path)
    return power_tensor, labels, subject_ids, brain_maps


def split_frequency_bands(brain_maps):
    return {
        "delta": brain_maps[:, :, :, 0:4],
        "theta": brain_maps[:, :, :, 4:8],
        "alpha": brain_maps[:, :, :, 8:12],
        "beta":  brain_maps[:, :, :, 12:35],
        "gamma": brain_maps[:, :, :, 35:40],
    }

## Modello: rete base con attention fra sub-band + rete siamese

`build_siamese` ora include `Dense(1) + Sigmoid` dopo la distanza
euclidea (variante stabile del BatchNorm+Sigmoid di Fig. 5 del paper) —
l'unico cambiamento architetturale rispetto alla versione precedente.

In [ ]:
def base_network_att(delta_shape, theta_shape, alpha_shape, beta_shape, gamma_shape,
                      embed_dim=16, num_heads=2, dropout_rate=0.1, l2_reg=1e-5):

    input_alpha = layers.Input(shape=alpha_shape, name="alpha_input")
    input_beta  = layers.Input(shape=beta_shape,  name="beta_input")
    input_delta = layers.Input(shape=delta_shape, name="delta_input")
    input_theta = layers.Input(shape=theta_shape, name="theta_input")
    input_gamma = layers.Input(shape=gamma_shape, name="gamma_input")

    lc_alpha = layers.LocallyConnected2D(1, kernel_size=5, activation="hard_sigmoid",
                                          kernel_regularizer=regularizers.l2(l2_reg))(input_alpha)
    lc_beta  = layers.LocallyConnected2D(1, kernel_size=5, activation="hard_sigmoid",
                                          kernel_regularizer=regularizers.l2(l2_reg))(input_beta)
    lc_delta = layers.LocallyConnected2D(1, kernel_size=5, activation="hard_sigmoid",
                                          kernel_regularizer=regularizers.l2(l2_reg))(input_delta)
    lc_theta = layers.LocallyConnected2D(1, kernel_size=5, activation="hard_sigmoid",
                                          kernel_regularizer=regularizers.l2(l2_reg))(input_theta)
    lc_gamma = layers.LocallyConnected2D(1, kernel_size=5, activation="hard_sigmoid",
                                          kernel_regularizer=regularizers.l2(l2_reg))(input_gamma)
    band_maps = [lc_alpha, lc_beta, lc_delta, lc_theta, lc_gamma]

    tokens = []
    for band_map in band_maps:
        pooled = layers.GlobalAveragePooling2D()(band_map)
        token = layers.Dense(embed_dim, activation="tanh",
                              kernel_regularizer=regularizers.l2(l2_reg))(pooled)
        token = layers.Dropout(dropout_rate)(token)
        tokens.append(token)

    token_seq = layers.Lambda(lambda t: tf.stack(t, axis=1))(tokens)

    attn_output, attn_scores = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim, name="band_attention"
    )(token_seq, token_seq, return_attention_scores=True)

    x = layers.Add()([token_seq, attn_output])
    x = layers.LayerNormalization()(x)

    ff = layers.Dense(embed_dim, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    ff = layers.Dropout(dropout_rate)(ff)
    ff = layers.Dense(embed_dim, kernel_regularizer=regularizers.l2(l2_reg))(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization()(x)

    importance_score = layers.Dense(1, activation="sigmoid")(x)
    importance_score = layers.Reshape((5,), name="importance_score")(importance_score)

    weighted_maps = []
    for i, band_map in enumerate(band_maps):
        weight = layers.Lambda(lambda t, idx=i: t[:, idx:idx+1])(importance_score)
        weight = layers.Reshape((1, 1, 1))(weight)
        weighted_maps.append(layers.Multiply()([band_map, weight]))

    merged = layers.Concatenate(axis=-1)(weighted_maps)

    y = layers.Conv2D(16, kernel_size=5, activation="tanh",
                       kernel_regularizer=regularizers.l2(l2_reg))(merged)
    y = layers.Conv2D(16, kernel_size=5, activation="tanh",
                       kernel_regularizer=regularizers.l2(l2_reg))(y)
    y = layers.Flatten()(y)
    y = layers.Dropout(dropout_rate)(y)
    embedding = layers.Dense(16, activation="tanh", name="embedding",
                              kernel_regularizer=regularizers.l2(l2_reg))(y)

    return Model(
        inputs=[input_alpha, input_beta, input_delta, input_theta, input_gamma],
        outputs=[embedding, importance_score],
        name="base_network_att"
    )


def build_siamese(base_network, alpha_shape, beta_shape, delta_shape, theta_shape, gamma_shape):
    alpha_A = layers.Input(alpha_shape)
    beta_A  = layers.Input(beta_shape)
    delta_A = layers.Input(delta_shape)
    theta_A = layers.Input(theta_shape)
    gamma_A = layers.Input(gamma_shape)

    alpha_B = layers.Input(alpha_shape)
    beta_B  = layers.Input(beta_shape)
    delta_B = layers.Input(delta_shape)
    theta_B = layers.Input(theta_shape)
    gamma_B = layers.Input(gamma_shape)

    embedded_A, _ = base_network([alpha_A, beta_A, delta_A, theta_A, gamma_A])
    embedded_B, _ = base_network([alpha_B, beta_B, delta_B, theta_B, gamma_B])

    # Distanza euclidea grezza, nessuna normalizzazione learned (Dense e
    # BatchNormalization hanno entrambe dato problemi: la prima collassava a
    # un output costante, la seconda disallineava training/inferenza). La
    # soglia di classificazione viene ora calibrata sul validation set di
    # ogni fold in train_fold(), invece di dipendere da una scala fissa.
    distance = layers.Lambda(euclidean_distance, name="distance")([embedded_A, embedded_B])

    model = Model(
        inputs=[[alpha_A, beta_A, delta_A, theta_A, gamma_A],
                [alpha_B, beta_B, delta_B, theta_B, gamma_B]],
        outputs=distance
    )
    return model

## Split LOOCV (train / val / test)

In [ ]:
def data_split(n_patients, labels, n_val=N_VAL_SUBJECTS, seed=42):
    """
    Split LOOCV con validation set STRATIFICATO per classe.

    Con n_val piccolo (4, come nel paper) uno split puramente casuale puo',
    per puro caso, produrre un validation set fatto di UNA SOLA classe.
    Con il pairing one-class questo azzera sia le coppie positive (richiedono
    entrambi ADHD) sia le negative (richiedono un ADHD + un Control), dando
    zero coppie di validazione totali -> Keras solleva
    "Expected input data to be non-empty".

    Per evitarlo, garantiamo sempre META' dei soggetti di validazione ADHD e
    META' Control (n_val=4 -> 2 ADHD + 2 Control).
    """
    rng = np.random.default_rng(seed)
    all_indices = np.arange(n_patients)
    n_val_per_class = n_val // 2
    folds = []

    for test_idx in range(n_patients):
        remaining = np.delete(all_indices, test_idx)
        remaining_labels = labels[remaining]

        adhd_pool = remaining[remaining_labels == 1]
        control_pool = remaining[remaining_labels == 0]

        adhd_shuffled = rng.permutation(adhd_pool)
        control_shuffled = rng.permutation(control_pool)

        val_idx = np.concatenate([
            adhd_shuffled[:n_val_per_class],
            control_shuffled[:n_val_per_class],
        ])
        train_idx = np.concatenate([
            adhd_shuffled[n_val_per_class:],
            control_shuffled[n_val_per_class:],
        ])
        rng.shuffle(train_idx)

        folds.append({
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": np.array([test_idx]),
        })
    return folds

## Pairing one-class (come nel paper, Fig. 9)

positive = solo ADHD-ADHD, negative = solo ADHD-Control. Le coppie
Control-Control sono scartate volutamente — è il design del paper, non un bug.

In [ ]:
def create_pairs(idx_array, labels):
    positive_indices = []
    negative_indices = []

    for i, j in combinations(range(len(idx_array)), 2):
        li, lj = labels[idx_array[i]], labels[idx_array[j]]
        if li == 1 and lj == 1:
            positive_indices.append([i, j])
        elif li != lj:
            negative_indices.append([i, j])
        # Control-Control: scartata volutamente, come nel paper

    positive_indices = np.array(positive_indices, dtype=int).reshape(-1, 2)
    negative_indices = np.array(negative_indices, dtype=int).reshape(-1, 2)
    return positive_indices, negative_indices

## Data augmentation (sub-band shuffling) + generator con wraparound

In [ ]:
def augment_batch(alpha, beta, delta, theta, gamma, subj_labels, idx, aug_fraction=AUG_FRACTION):
    bands = {"alpha": alpha, "beta": beta, "delta": delta, "theta": theta, "gamma": gamma}
    n = len(idx)
    out = {name: arr[idx].copy() for name, arr in bands.items()}
    labels_here = subj_labels[idx]

    for band_i, name in enumerate(bands.keys()):
        start = int(band_i * aug_fraction * n)
        end = int((band_i + 1) * aug_fraction * n)
        for pos in range(start, min(end, n)):
            same_class_pool = np.where(subj_labels == labels_here[pos])[0]
            donor = np.random.choice(same_class_pool)
            out[name][pos] = bands[name][donor]

    return out["alpha"], out["beta"], out["delta"], out["theta"], out["gamma"]


def data_generator(epoch_number, positive_indices, negative_indices,
                    train_alpha, train_beta, train_delta, train_theta, train_gamma,
                    train_labels, positive_batch_size, negative_batch_size, steps_per_epoch):

    n_pos, n_neg = len(positive_indices), len(negative_indices)

    for epoch in range(epoch_number):
        np.random.shuffle(positive_indices)
        np.random.shuffle(negative_indices)
        pos_pointer = neg_pointer = 0

        for step in range(steps_per_epoch):
            if pos_pointer + positive_batch_size > n_pos:
                np.random.shuffle(positive_indices)
                pos_pointer = 0
            if neg_pointer + negative_batch_size > n_neg:
                np.random.shuffle(negative_indices)
                neg_pointer = 0

            pos_pairs = positive_indices[pos_pointer:pos_pointer + positive_batch_size]
            neg_pairs = negative_indices[neg_pointer:neg_pointer + negative_batch_size]

            x1_idx = np.concatenate([pos_pairs[:, 0], neg_pairs[:, 0]])
            x2_idx = np.concatenate([pos_pairs[:, 1], neg_pairs[:, 1]])

            x1 = augment_batch(train_alpha, train_beta, train_delta, train_theta, train_gamma,
                                train_labels, x1_idx)
            x2 = augment_batch(train_alpha, train_beta, train_delta, train_theta, train_gamma,
                                train_labels, x2_idx)

            y = np.concatenate([np.ones(len(pos_pairs)), np.zeros(len(neg_pairs))])

            yield [list(x1), list(x2)], y

            pos_pointer += positive_batch_size
            neg_pointer += negative_batch_size

## Training per fold (invariato)

In [ ]:
def train_fold(fold, bands, labels, epochs=50, positive_batch_size=16,
               negative_batch_size=16, steps_per_epoch=100, patience=6, verbose=0):

    train_idx, val_idx = fold["train_idx"], fold["val_idx"]

    alpha_maps, beta_maps = bands["alpha"], bands["beta"]
    delta_maps, theta_maps, gamma_maps = bands["delta"], bands["theta"], bands["gamma"]

    shapes = {
        "delta": delta_maps.shape[1:], "theta": theta_maps.shape[1:],
        "alpha": alpha_maps.shape[1:], "beta": beta_maps.shape[1:],
        "gamma": gamma_maps.shape[1:],
    }

    base_network = base_network_att(
        delta_shape=shapes["delta"], theta_shape=shapes["theta"],
        alpha_shape=shapes["alpha"], beta_shape=shapes["beta"], gamma_shape=shapes["gamma"],
    )
    siamese_model = build_siamese(base_network, shapes["alpha"], shapes["beta"],
                                   shapes["delta"], shapes["theta"], shapes["gamma"])
    siamese_model.compile(optimizer="adam", loss=contrastive_loss)

    positive_indices, negative_indices = create_pairs(train_idx, labels)

    train_alpha, train_beta = alpha_maps[train_idx], beta_maps[train_idx]
    train_delta, train_theta, train_gamma = delta_maps[train_idx], theta_maps[train_idx], gamma_maps[train_idx]
    train_labels_rel = labels[train_idx]

    val_pos, val_neg = create_pairs(val_idx, labels)
    val_alpha, val_beta = alpha_maps[val_idx], beta_maps[val_idx]
    val_delta, val_theta, val_gamma = delta_maps[val_idx], theta_maps[val_idx], gamma_maps[val_idx]

    val_x1_idx = np.concatenate([val_pos[:, 0], val_neg[:, 0]])
    val_x2_idx = np.concatenate([val_pos[:, 1], val_neg[:, 1]])

    val_x1 = [val_alpha[val_x1_idx], val_beta[val_x1_idx], val_delta[val_x1_idx],
              val_theta[val_x1_idx], val_gamma[val_x1_idx]]
    val_x2 = [val_alpha[val_x2_idx], val_beta[val_x2_idx], val_delta[val_x2_idx],
              val_theta[val_x2_idx], val_gamma[val_x2_idx]]
    val_y = np.concatenate([np.ones(len(val_pos)), np.zeros(len(val_neg))])

    generator = data_generator(
        epoch_number=epochs,
        positive_indices=positive_indices, negative_indices=negative_indices,
        train_alpha=train_alpha, train_beta=train_beta, train_delta=train_delta,
        train_theta=train_theta, train_gamma=train_gamma, train_labels=train_labels_rel,
        positive_batch_size=positive_batch_size, negative_batch_size=negative_batch_size,
        steps_per_epoch=steps_per_epoch,
    )

    early_stop = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)

    history = siamese_model.fit(
        generator, epochs=epochs, steps_per_epoch=steps_per_epoch,
        validation_data=([val_x1, val_x2], val_y),
        callbacks=[early_stop], verbose=verbose,
    )

    # Niente calibrazione soglia: con un validation set di soli 4 soggetti
    # (2 ADHD + 2 Control, come nel paper) ci sono solo ~5 coppie totali per
    # fold -> calibrare una soglia su cosi' pochi dati e' overfitting, non
    # stima. Torniamo a margin/2 fisso, come fa il paper, dando pero' al
    # training piu' epoche (vedi DEBUG_EPOCHS/FULL_EPOCHS) perche' la
    # contrastive loss abbia il tempo di spingere le distanze verso la scala
    # attesa dal margin.
    return siamese_model, history.history

## Valutazione (majority vote solo contro riferimenti ADHD, come nel paper)

Torniamo alla distanza euclidea grezza e alla soglia fissa `margin/2 = 0.5`,
esattamente come nel paper. Niente calibrazione sul validation set: con soli
4 soggetti di validazione (2 ADHD + 2 Control) le coppie disponibili sono
troppo poche (~5) per stimare una soglia in modo affidabile — calibrare su
così pochi dati produce overfitting, non una stima. Diamo invece al training
più epoche (`DEBUG_EPOCHS`/`FULL_EPOCHS`), così la contrastive loss ha il
tempo di spingere le distanze verso la scala attesa dal margin.

In [ ]:
def evaluate_fold_majority_vote(fold, bands, labels, subject_ids, siamese_model,
                                 margin=CONTRASTIVE_MARGIN):
    train_idx, test_idx = fold["train_idx"], fold["test_idx"]
    alpha_maps, beta_maps = bands["alpha"], bands["beta"]
    delta_maps, theta_maps, gamma_maps = bands["delta"], bands["theta"], bands["gamma"]

    adhd_train_idx = train_idx[labels[train_idx] == 1]
    n_ref = len(adhd_train_idx)

    test_inputs = [alpha_maps[test_idx], beta_maps[test_idx], delta_maps[test_idx],
                   theta_maps[test_idx], gamma_maps[test_idx]]
    ref_inputs = [alpha_maps[adhd_train_idx], beta_maps[adhd_train_idx], delta_maps[adhd_train_idx],
                  theta_maps[adhd_train_idx], gamma_maps[adhd_train_idx]]

    x1 = [np.repeat(t, n_ref, axis=0) for t in test_inputs]
    x2 = ref_inputs

    distances = siamese_model.predict([x1, x2], verbose=0).ravel()

    votes = (distances < margin / 2).astype(int)
    score = votes.mean()
    predicted_label = int(score > 0.5)
    true_label = int(labels[test_idx][0])

    return predicted_label, true_label, score, subject_ids[test_idx][0]

## Caricamento dati ed esecuzione split

In [ ]:
power_tensor, labels, subject_ids, brain_maps = load_precomputed(
    power_tensor_path="power_tensor.npy",
    labels_path="labels.npy",
    subject_ids_path="subject_ids.npy",
    brain_maps_path="brain_maps.npy",
)
bands = split_frequency_bands(brain_maps)

n_patients = len(labels)
folds = data_split(n_patients=n_patients, labels=labels, n_val=N_VAL_SUBJECTS, seed=42)
print("Numero di fold:", len(folds))

## Debug rapido (consigliato prima del run completo)

Esegue solo `N_DEBUG_FOLDS` fold, scelti a metà tra soggetti test ADHD e
Control, con training ridotto. Imposta `N_DEBUG_FOLDS = None` sopra per
saltare questa cella.

In [ ]:
if N_DEBUG_FOLDS is not None:
    adhd_fold_idx = [i for i, f in enumerate(folds) if labels[f["test_idx"][0]] == 1]
    control_fold_idx = [i for i, f in enumerate(folds) if labels[f["test_idx"][0]] == 0]

    rng = np.random.default_rng(0)
    n_each = N_DEBUG_FOLDS // 2
    debug_fold_idx = list(rng.choice(adhd_fold_idx, size=n_each, replace=False)) + \
                      list(rng.choice(control_fold_idx, size=n_each, replace=False))

    print(f"Fold di debug scelti: {debug_fold_idx}\n")

    dbg_true, dbg_pred, dbg_score = [], [], []

    for i in debug_fold_idx:
        fold = folds[i]
        siamese_model, history = train_fold(
            fold, bands, labels,
            epochs=DEBUG_EPOCHS, steps_per_epoch=DEBUG_STEPS_PER_EPOCH,
            patience=DEBUG_PATIENCE, verbose=0,
        )
        pred, true, score, subj = evaluate_fold_majority_vote(
            fold, bands, labels, subject_ids, siamese_model
        )
        dbg_true.append(true); dbg_pred.append(pred); dbg_score.append(score)

        correct = "OK" if pred == true else "SBAGLIATO"
        print(f"Fold {i} - soggetto {subj}: vero={true} pred={pred} score={score:.3f} [{correct}]  "
              f"(train_loss={history['loss'][-1]:.4f}, val_loss={history['val_loss'][-1]:.4f}, "
              f"epoche={len(history['loss'])}/{DEBUG_EPOCHS})")

    dbg_true, dbg_pred, dbg_score = np.array(dbg_true), np.array(dbg_pred), np.array(dbg_score)
    print("\n" + "=" * 60)
    print(f"Accuracy debug: {(dbg_true == dbg_pred).mean():.3f}")
    if len(set(dbg_true.tolist())) > 1:
        print(f"AUC debug: {roc_auc_score(dbg_true, dbg_score):.3f}")
    print(f"Frazione predizioni ADHD: {dbg_pred.mean():.3f}  (atteso ~0.5 se bilanciato)")
    print("=" * 60)

## Run completo LOOCV (121 fold — lungo, esegui solo dopo aver validato il debug)

In [ ]:
y_true, y_pred, y_score, subj_out = [], [], [], []

for i, fold in enumerate(folds):
    siamese_model, history = train_fold(
        fold, bands, labels, epochs=FULL_EPOCHS,
        steps_per_epoch=FULL_STEPS_PER_EPOCH, patience=FULL_PATIENCE, verbose=0
    )
    pred, true, score, subj = evaluate_fold_majority_vote(
        fold, bands, labels, subject_ids, siamese_model
    )

    y_true.append(true)
    y_pred.append(pred)
    y_score.append(score)
    subj_out.append(subj)

    print(f"Fold {i + 1}/{len(folds)} - subject {subj}: pred={pred} true={true} score={score:.3f}")

## Metriche finali

In [ ]:
acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
auc = roc_auc_score(y_true, y_score)

print(f"\nLOOCV Accuracy: {acc:.3f}")
print(f"LOOCV AUC: {auc:.3f}")
print("Confusion matrix:\n", cm)
print(f"Frazione predizioni ADHD: {np.mean(y_pred):.3f}  (atteso ~0.5 se bilanciato)")

## Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - LOOCV (majority vote, come nel paper)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Importanza delle bande (attention) — ADHD vs Control

Il blocco attention dentro `base_network_att` calcola già `importance_score`
per ciascuna delle 5 sub-band ad ogni forward pass — è la vostra estensione
rispetto al paper, che invece doveva ricorrere a Grad-CAM a posteriori per
capire quali regioni/bande contassero di più.

Qui alleniamo un fold, estraiamo `base_network` dal `siamese_model` allenato
(è condiviso tra i due rami — lo recuperiamo per nome), e lo usiamo
direttamente su un soggetto ADHD e uno Control per confrontare visivamente
i pesi di importanza per banda. Nessun Grad-CAM necessario.

In [ ]:

# Alleniamo un fold di esempio (puoi cambiare quale)
example_fold = folds[0]
siamese_model_ex, history_ex = train_fold(
    example_fold, bands, labels,
    epochs=DEBUG_EPOCHS, steps_per_epoch=DEBUG_STEPS_PER_EPOCH,
    patience=DEBUG_PATIENCE, verbose=0,
)

# base_network e' condivisa tra i due rami del siamese: la recuperiamo per nome
base_network_ex = siamese_model_ex.get_layer("base_network_att")

# Un soggetto ADHD e uno Control presi dal train set del fold (qualsiasi soggetto va bene)
train_idx = example_fold["train_idx"]
adhd_idx_ex = train_idx[labels[train_idx] == 1][0]
control_idx_ex = train_idx[labels[train_idx] == 0][0]

def get_importance(subj_idx):
    inputs = [
        bands["alpha"][subj_idx:subj_idx+1], bands["beta"][subj_idx:subj_idx+1],
        bands["delta"][subj_idx:subj_idx+1], bands["theta"][subj_idx:subj_idx+1],
        bands["gamma"][subj_idx:subj_idx+1],
    ]
    _, importance = base_network_ex.predict(inputs, verbose=0)
    return importance[0]  # shape (5,): alpha, beta, delta, theta, gamma

importance_adhd = get_importance(adhd_idx_ex)
importance_control = get_importance(control_idx_ex)

band_names = ["Alpha", "Beta", "Delta", "Theta", "Gamma"]
x = np.arange(len(band_names))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, importance_adhd, width, label=f"ADHD ({subject_ids[adhd_idx_ex]})", color="tab:red")
ax.bar(x + width/2, importance_control, width, label=f"Control ({subject_ids[control_idx_ex]})", color="tab:blue")
ax.set_xticks(x)
ax.set_xticklabels(band_names)
ax.set_ylabel("Importance score (0-1)")
ax.set_title("Importanza delle sub-band per soggetto (dal blocco attention)")
ax.legend()
plt.tight_layout()
plt.show()

print("Importance ADHD:  ", dict(zip(band_names, importance_adhd.round(3))))
print("Importance Control:", dict(zip(band_names, importance_control.round(3))))
